# Variant Tables EDA - Drug Response, Cancer, Population
**DNA Gene Mapping Project - ML Phase V5**  
**Author:** Sharique Mohammad  
**Date:** February 2026  
**Tables covered:**
- variant_cancer_ml_features (3.1M variants, 50 cols)
- variant_drug_response_ml_features (4.1M variants, 57 cols)
- variant_population_ml_features (46K variants, 51 cols)
- drug_response_ml_features (4.1M variants, 31 cols)

## Objective
EDA on the four variant-level tables covering cancer, drug response, and population frequency use cases. Identify class imbalance, target variable distributions, and key feature patterns.

## Use Cases Covered
- UC6: Drug Response Variant Priority (target: is_actionable_pharmacogene_variant)
- UC7: Cancer Variant Classification (target: is_driver_candidate)
- UC8: Population Carrier Screening (target: is_carrier_screening_candidate)
- UC9: Variant Population Frequency Risk (target: is_clinically_actionable_rare_variant)

## Deliverables
- Visualizations saved per table under respective analytical folders
- EDA reports per table
- Missing values, correlation, feature statistics per table

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path().absolute().parent.parent
ANALYTICAL   = PROJECT_ROOT / 'data' / 'analytical'

def make_dirs(table_name):
    base = ANALYTICAL / table_name
    (base / 'images').mkdir(parents=True, exist_ok=True)
    (base / 'reports').mkdir(parents=True, exist_ok=True)
    (base / 'metrics').mkdir(parents=True, exist_ok=True)
    return base / 'images', base / 'reports', base / 'metrics'

print("Setup complete")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")
print(f"Host : {POSTGRES_HOST}:{POSTGRES_PORT}")
print(f"DB   : {POSTGRES_DB}")

---
## 3. variant_cancer_ml_features
**Use Case 7 — Cancer Variant Classification**
**Target:** is_driver_candidate

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('variant_cancer_ml_features')

print("Loading variant_cancer_ml_features (10% sample)...")
df_cancer = pd.read_sql(
    "SELECT * FROM gold.variant_cancer_ml_features TABLESAMPLE SYSTEM (10)",
    engine
)
print(f"Rows: {len(df_cancer):,}  Cols: {len(df_cancer.columns)}")

int_cols = [
    'position', 'sample_count', 'total_mutation_count', 'missense_sample_count',
    'truncating_sample_count', 'silent_sample_count', 'snv_sample_count',
    'indel_sample_count', 'gene_total_samples', 'gene_unique_sites',
    'cancer_mutation_burden_score', 'functional_impact_prediction',
    'tissue_expression_in_tumors', 'has_kinase_domain_count',
    'driver_likelihood_score', 'therapeutic_target_score'
]
double_cols = [
    'cancer_priority_score', 'conservation_score', 'cadd_phred',
    'max_tumor_expression', 'germline_variant_frequency', 'prognostic_value_score'
]
bool_cols = [
    'is_recurrent_mutation', 'is_hotspot_mutation', 'is_high_impact_cancer_variant',
    'is_driver_candidate', 'is_cancer_gene', 'is_tumor_suppressor_candidate',
    'is_oncogene_candidate', 'clinvar_is_pathogenic', 'cancer_disease_associations',
    'hereditary_cancer_syndrome', 'affected_oncogenic_domains',
    'kinase_domain_mutations', 'is_rare', 'is_kinase', 'is_receptor',
    'is_enzyme', 'is_pharmacogene'
]

for col in int_cols:
    if col in df_cancer.columns:
        df_cancer[col] = pd.to_numeric(df_cancer[col], errors='coerce').astype('Int64')
for col in double_cols:
    if col in df_cancer.columns:
        df_cancer[col] = pd.to_numeric(df_cancer[col], errors='coerce')
for col in bool_cols:
    if col in df_cancer.columns:
        df_cancer[col] = df_cancer[col].astype(str).str.lower().map({'true': True, 'false': False})

print("Types converted")

In [ ]:
total_c = len(df_cancer)

driver    = int(df_cancer['is_driver_candidate'].sum())           if 'is_driver_candidate'           in df_cancer.columns else 0
hotspot   = int(df_cancer['is_hotspot_mutation'].sum())           if 'is_hotspot_mutation'            in df_cancer.columns else 0
recurrent = int(df_cancer['is_recurrent_mutation'].sum())         if 'is_recurrent_mutation'          in df_cancer.columns else 0
high_imp  = int(df_cancer['is_high_impact_cancer_variant'].sum()) if 'is_high_impact_cancer_variant'  in df_cancer.columns else 0

print("variant_cancer_ml_features - Target Variable Summary")
print("=" * 55)
print(f"Driver Candidate    : {driver:>10,}  ({driver/total_c*100:.1f}%)")
print(f"Hotspot Mutation    : {hotspot:>10,}  ({hotspot/total_c*100:.1f}%)")
print(f"Recurrent Mutation  : {recurrent:>10,}  ({recurrent/total_c*100:.1f}%)")
print(f"High Impact Cancer  : {high_imp:>10,}  ({high_imp/total_c*100:.1f}%)")

if driver > 0 and (total_c - driver) > 0:
    ratio = max(driver, total_c - driver) / min(driver, total_c - driver)
    print(f"\nClass imbalance (is_driver_candidate) : {ratio:.2f}:1")
    print(f"SMOTE needed                           : {ratio > 5}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

target_vals = ['Driver', 'Hotspot', 'Recurrent', 'High Impact']
target_cnts = [driver, hotspot, recurrent, high_imp]
colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6']
axes[0].bar(target_vals, target_cnts, color=colors, alpha=0.8, edgecolor='black')
for i, (name, val) in enumerate(zip(target_vals, target_cnts)):
    axes[0].text(i, val, f'{val:,}\n({val/total_c*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('Cancer Variant Target Variables', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', alpha=0.3)

if 'mutation_frequency_category' in df_cancer.columns:
    mfc_dist = df_cancer['mutation_frequency_category'].value_counts()
    mfc_dist.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Mutation Frequency Category', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variables.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variables.png")

In [ ]:
if 'gene_cancer_role' in df_cancer.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    role_dist = df_cancer['gene_cancer_role'].value_counts()
    role_dist.sort_values().plot(kind='barh', ax=axes[0], color='coral', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[0].set_title('Gene Cancer Role Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    if 'cancer_priority_score' in df_cancer.columns:
        axes[1].hist(df_cancer['cancer_priority_score'].dropna(), bins=40,
                     color='darkred', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('Cancer Priority Score', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
        axes[1].set_title('Cancer Priority Score Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02_gene_cancer_role.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 02_gene_cancer_role.png")

miss_c = pd.DataFrame({'column': df_cancer.columns,
                        'missing_pct': (df_cancer.isnull().sum().values / total_c * 100).round(2)
                       }).sort_values('missing_pct', ascending=False)
miss_c.to_csv(METRICS_DIR / 'missing_values.csv', index=False)

corr_f = [c for c in ['sample_count','total_mutation_count','missense_sample_count',
           'gene_total_samples','cancer_mutation_burden_score',
           'driver_likelihood_score','therapeutic_target_score',
           'cancer_priority_score','conservation_score','cadd_phred'] if c in df_cancer.columns]
corr_m = df_cancer[corr_f].apply(pd.to_numeric, errors='coerce').corr()
corr_m.to_csv(METRICS_DIR / 'correlation_matrix.csv')
df_cancer[corr_f].describe().T.to_csv(METRICS_DIR / 'feature_statistics.csv')

print(f"Saved metrics to {METRICS_DIR}")

---
## 4. variant_drug_response_ml_features
**Use Case 6 — Drug Response Variant Priority**
**Target:** is_actionable_pharmacogene_variant

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('variant_drug_response_ml_features')

print("Loading variant_drug_response_ml_features (10% sample)...")
df_dr = pd.read_sql(
    "SELECT * FROM gold.variant_drug_response_ml_features TABLESAMPLE SYSTEM (10)",
    engine
)
print(f"Rows: {len(df_dr):,}  Cols: {len(df_dr.columns)}")

int_cols_dr = [
    'conservation_level', 'pathogenicity_score', 'mutation_severity_score',
    'tissues_expressed_count', 'total_disease_count', 'cancer_mutation_count',
    'pharmacogene_annotation_score', 'functional_impact_score',
    'population_adjusted_score', 'tissue_specific_response_score'
]
double_cols_dr = [
    'phylop_score', 'cadd_phred', 'max_expression_tpm',
    'allele_frequency', 'druggability_score', 'drug_response_priority_score'
]
bool_cols_dr = [
    'is_pathogenic', 'is_benign', 'is_vus', 'is_missense_variant',
    'is_frameshift_variant', 'is_nonsense_variant', 'is_splice_variant',
    'has_functional_domain', 'affects_functional_domain', 'has_pharmgkb_annotation',
    'has_high_conservation', 'affects_drug_metabolism', 'affects_drug_efficacy',
    'is_high_impact_variant', 'is_hepatic_drug_metabolism_variant',
    'is_common_pharmacogene_variant', 'is_potential_resistance_variant',
    'is_liver_expressed', 'is_common_variant', 'is_rare_variant',
    'has_cancer_disease', 'has_cardiovascular_disease', 'has_neurological_disease',
    'is_cancer_gene', 'is_pharmacogene', 'is_actionable_pharmacogene_variant',
    'indication_specific_actionability'
]

for col in int_cols_dr:
    if col in df_dr.columns:
        df_dr[col] = pd.to_numeric(df_dr[col], errors='coerce').astype('Int64')
for col in double_cols_dr:
    if col in df_dr.columns:
        df_dr[col] = pd.to_numeric(df_dr[col], errors='coerce')
for col in bool_cols_dr:
    if col in df_dr.columns:
        df_dr[col] = df_dr[col].astype(str).str.lower().map({'true': True, 'false': False})

total_dr = len(df_dr)

actionable = int(df_dr['is_actionable_pharmacogene_variant'].sum()) if 'is_actionable_pharmacogene_variant' in df_dr.columns else 0
print(f"\nTarget: is_actionable_pharmacogene_variant")
print(f"Actionable     : {actionable:,} ({actionable/total_dr*100:.1f}%)")
print(f"Non-actionable : {total_dr - actionable:,} ({(total_dr-actionable)/total_dr*100:.1f}%)")
if actionable > 0 and (total_dr - actionable) > 0:
    ratio = max(actionable, total_dr-actionable) / min(actionable, total_dr-actionable)
    print(f"Imbalance      : {ratio:.2f}:1  SMOTE: {ratio > 5}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['Actionable', 'Non-Actionable'],
            [actionable, total_dr - actionable],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([actionable, total_dr - actionable]):
    axes[0].text(i, val, f'{val:,}\n({val/total_dr*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Drug Response Target Variable', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'drug_response_category' in df_dr.columns:
    drc_dist = df_dr['drug_response_category'].value_counts()
    drc_dist.sort_values().plot(kind='barh', ax=axes[1], color='teal', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Drug Response Category Distribution', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variable.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variable.png")

drug_flags = ['affects_drug_metabolism', 'affects_drug_efficacy',
              'is_hepatic_drug_metabolism_variant', 'is_potential_resistance_variant',
              'is_common_pharmacogene_variant', 'has_pharmgkb_annotation']
drug_flags = [c for c in drug_flags if c in df_dr.columns]
drug_counts = {col: int(df_dr[col].sum()) for col in drug_flags}

fig, ax = plt.subplots(figsize=(12, 6))
names  = [c.replace('_', ' ').title() for c in drug_flags]
values = [drug_counts[c] for c in drug_flags]
bars = ax.barh(names, values, color='mediumpurple', alpha=0.8, edgecolor='black')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values)*0.01, bar.get_y() + bar.get_height()/2.,
            f'{val/total_dr*100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Count', fontsize=11, fontweight='bold')
ax.set_title('Drug Response Feature Flags', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(values) * 1.2)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_drug_response_flags.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_drug_response_flags.png")

miss_dr = pd.DataFrame({'column': df_dr.columns,
                         'missing_pct': (df_dr.isnull().sum().values / total_dr * 100).round(2)
                        }).sort_values('missing_pct', ascending=False)
miss_dr.to_csv(METRICS_DIR / 'missing_values.csv', index=False)

corr_f_dr = [c for c in ['conservation_level','pathogenicity_score','mutation_severity_score',
             'pharmacogene_annotation_score','functional_impact_score',
             'phylop_score','cadd_phred','allele_frequency','druggability_score',
             'drug_response_priority_score'] if c in df_dr.columns]
df_dr[corr_f_dr].apply(pd.to_numeric, errors='coerce').corr().to_csv(METRICS_DIR / 'correlation_matrix.csv')
df_dr[corr_f_dr].describe().T.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved metrics to {METRICS_DIR}")

---
## 5. variant_population_ml_features
**Use Cases 8 and 9 — Carrier Screening and Population Frequency Risk**
**Targets:** is_carrier_screening_candidate, is_clinically_actionable_rare_variant

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('variant_population_ml_features')

print("Loading variant_population_ml_features (full table, ~46K rows)...")
df_pop = pd.read_sql("SELECT * FROM gold.variant_population_ml_features", engine)
print(f"Rows: {len(df_pop):,}  Cols: {len(df_pop.columns)}")

int_cols_pop = [
    'rarity_score', 'carrier_risk_score', 'pathogenicity_likelihood_score',
    'pathogenicity_score', 'conservation_level', 'total_gene_variants',
    'lof_variants', 'total_disease_count', 'somatic_frequency',
    'expression_tissues', 'pathogenicity_likelihood_refined'
]
double_cols_pop = [
    'allele_frequency', 'gene_constraint_score',
    'clinical_significance_frequency_score', 'carrier_risk_score_adjusted'
]
bool_cols_pop = [
    'is_ultra_rare_variant', 'is_very_rare_variant', 'is_rare_variant',
    'is_low_frequency_variant', 'is_common_variant',
    'is_pathogenic', 'is_benign', 'is_vus',
    'is_germline', 'is_somatic', 'clinvar_pathogenic', 'clinvar_benign',
    'pathogenicity_frequency_conflict', 'rare_pathogenic_variant',
    'common_benign_validation', 'germline_cancer_predisposition',
    'tissue_specific_allele_effects',
    'is_clinically_actionable_rare_variant', 'is_carrier_screening_candidate'
]

for col in int_cols_pop:
    if col in df_pop.columns:
        df_pop[col] = pd.to_numeric(df_pop[col], errors='coerce').astype('Int64')
for col in double_cols_pop:
    if col in df_pop.columns:
        df_pop[col] = pd.to_numeric(df_pop[col], errors='coerce')
for col in bool_cols_pop:
    if col in df_pop.columns:
        df_pop[col] = df_pop[col].astype(str).str.lower().map({'true': True, 'false': False})

total_pop = len(df_pop)

actionable_rare = int(df_pop['is_clinically_actionable_rare_variant'].sum()) if 'is_clinically_actionable_rare_variant' in df_pop.columns else 0
carrier_cand    = int(df_pop['is_carrier_screening_candidate'].sum())         if 'is_carrier_screening_candidate'         in df_pop.columns else 0

print(f"\nUC9 Target - is_clinically_actionable_rare_variant : {actionable_rare:,} ({actionable_rare/total_pop*100:.1f}%)")
print(f"UC8 Target - is_carrier_screening_candidate         : {carrier_cand:,} ({carrier_cand/total_pop*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['Actionable Rare', 'Not Actionable'],
            [actionable_rare, total_pop - actionable_rare],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([actionable_rare, total_pop - actionable_rare]):
    axes[0].text(i, val, f'{val:,}\n({val/total_pop*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('UC9: Clinically Actionable Rare Variant', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(['Carrier Candidate', 'Not Candidate'],
            [carrier_cand, total_pop - carrier_cand],
            color=['#f39c12', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([carrier_cand, total_pop - carrier_cand]):
    axes[1].text(i, val, f'{val:,}\n({val/total_pop*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('UC8: Carrier Screening Candidate', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variables.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variables.png")

if 'frequency_tier' in df_pop.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    freq_tier_dist = df_pop['frequency_tier'].value_counts()
    freq_tier_dist.sort_values().plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[0].set_title('Frequency Tier Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    if 'allele_frequency' in df_pop.columns:
        freq_data = df_pop['allele_frequency'].dropna()
        axes[1].hist(np.log10(freq_data[freq_data > 0] + 1e-9), bins=50,
                     color='navy', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('log10(Allele Frequency)', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
        axes[1].set_title('Allele Frequency Distribution (log scale)', fontsize=12, fontweight='bold')
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02_frequency_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 02_frequency_distribution.png")

miss_pop = pd.DataFrame({'column': df_pop.columns,
                          'missing_pct': (df_pop.isnull().sum().values / total_pop * 100).round(2)
                         }).sort_values('missing_pct', ascending=False)
miss_pop.to_csv(METRICS_DIR / 'missing_values.csv', index=False)

corr_f_pop = [c for c in ['rarity_score','carrier_risk_score','pathogenicity_likelihood_score',
              'pathogenicity_score','conservation_level','allele_frequency',
              'gene_constraint_score','clinical_significance_frequency_score'] if c in df_pop.columns]
df_pop[corr_f_pop].apply(pd.to_numeric, errors='coerce').corr().to_csv(METRICS_DIR / 'correlation_matrix.csv')
df_pop[corr_f_pop].describe().T.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved metrics to {METRICS_DIR}")

---
## 6. drug_response_ml_features
**Use Case 14 — Drug Response Variant Priority (source table)**
**Target:** is_actionable_pharmacogene_variant

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('drug_response_ml_features')

print("Loading drug_response_ml_features (10% sample)...")
df_drf = pd.read_sql(
    "SELECT * FROM gold.drug_response_ml_features TABLESAMPLE SYSTEM (10)",
    engine
)
print(f"Rows: {len(df_drf):,}  Cols: {len(df_drf.columns)}")

int_cols_drf = ['conservation_level', 'pharmacogene_annotation_score',
                'functional_impact_score', 'pathogenicity_score']
double_cols_drf = ['phylop_score', 'cadd_phred', 'drug_response_priority_score']
bool_cols_drf = [
    'is_pathogenic', 'is_benign', 'is_vus', 'is_missense_variant',
    'is_frameshift_variant', 'is_nonsense_variant', 'is_splice_variant',
    'has_functional_domain', 'affects_functional_domain', 'has_pharmgkb_annotation',
    'has_high_conservation', 'affects_drug_metabolism', 'affects_drug_efficacy',
    'is_high_impact_variant', 'is_actionable_pharmacogene_variant'
]

for col in int_cols_drf:
    if col in df_drf.columns:
        df_drf[col] = pd.to_numeric(df_drf[col], errors='coerce').astype('Int64')
for col in double_cols_drf:
    if col in df_drf.columns:
        df_drf[col] = pd.to_numeric(df_drf[col], errors='coerce')
for col in bool_cols_drf:
    if col in df_drf.columns:
        df_drf[col] = df_drf[col].astype(str).str.lower().map({'true': True, 'false': False})

total_drf = len(df_drf)

actionable_drf = int(df_drf['is_actionable_pharmacogene_variant'].sum()) if 'is_actionable_pharmacogene_variant' in df_drf.columns else 0
print(f"\nTarget: is_actionable_pharmacogene_variant")
print(f"Actionable     : {actionable_drf:,} ({actionable_drf/total_drf*100:.1f}%)")
if actionable_drf > 0 and (total_drf - actionable_drf) > 0:
    ratio_drf = max(actionable_drf, total_drf-actionable_drf) / min(actionable_drf, total_drf-actionable_drf)
    print(f"Imbalance      : {ratio_drf:.2f}:1  SMOTE: {ratio_drf > 5}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['Actionable', 'Non-Actionable'],
            [actionable_drf, total_drf - actionable_drf],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([actionable_drf, total_drf - actionable_drf]):
    axes[0].text(i, val, f'{val:,}\n({val/total_drf*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Drug Response Source Table - Target Variable', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'drug_response_priority' in df_drf.columns:
    drp_dist = df_drf['drug_response_priority'].value_counts()
    drp_dist.sort_values().plot(kind='barh', ax=axes[1], color='teal', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Drug Response Priority Distribution', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variable.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variable.png")

if 'drug_response_priority_score' in df_drf.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(df_drf['drug_response_priority_score'].dropna(), bins=40,
            color='mediumpurple', alpha=0.8, edgecolor='black')
    ax.set_xlabel('Drug Response Priority Score', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title('Drug Response Priority Score Distribution', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02_priority_score.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 02_priority_score.png")

miss_drf = pd.DataFrame({'column': df_drf.columns,
                          'missing_pct': (df_drf.isnull().sum().values / total_drf * 100).round(2)
                         }).sort_values('missing_pct', ascending=False)
miss_drf.to_csv(METRICS_DIR / 'missing_values.csv', index=False)

corr_f_drf = [c for c in ['conservation_level','pharmacogene_annotation_score',
               'functional_impact_score','pathogenicity_score',
               'phylop_score','cadd_phred','drug_response_priority_score'] if c in df_drf.columns]
df_drf[corr_f_drf].apply(pd.to_numeric, errors='coerce').corr().to_csv(METRICS_DIR / 'correlation_matrix.csv')
df_drf[corr_f_drf].describe().T.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved metrics to {METRICS_DIR}")

## 7. Summary

In [ ]:
print("=" * 65)
print("VARIANT TABLES EDA COMPLETE")
print("=" * 65)
print()
print("Tables analyzed:")
print(f"  variant_cancer_ml_features         : {total_c:,} rows (10% sample)")
print(f"  variant_drug_response_ml_features  : {total_dr:,} rows (10% sample)")
print(f"  variant_population_ml_features     : {total_pop:,} rows (full)")
print(f"  drug_response_ml_features          : {total_drf:,} rows (10% sample)")
print()
print("Target variable summary:")
print(f"  UC7 is_driver_candidate                  : {driver/total_c*100:.1f}% positive")
print(f"  UC6 is_actionable_pharmacogene_variant   : {actionable/total_dr*100:.1f}% positive (variant_dr)")
print(f"  UC14 is_actionable_pharmacogene_variant  : {actionable_drf/total_drf*100:.1f}% positive (drug_response)")
print(f"  UC9 is_clinically_actionable_rare_variant: {actionable_rare/total_pop*100:.1f}% positive")
print(f"  UC8 is_carrier_screening_candidate       : {carrier_cand/total_pop*100:.1f}% positive")
print()
print("Next: 07_population_cancer_eda.ipynb")